<a href="https://colab.research.google.com/github/manishashivarathri2004-ctrl/asr-error-correction/blob/main/recentfinal.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install tensorflow flask pyngrok scikit-learn pandas numpy


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from flask import Flask, request, render_template_string
from pyngrok import ngrok
import os

In [ ]:
!wget https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data

In [ ]:
column_names = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'num']
df = pd.read_csv('processed.cleveland.data', names=column_names)
print(df.head())  # View first 5 rows
print(df.info())  # Check data types and missing values

In [ ]:
df.replace('?', np.nan, inplace=True)
df.dropna(inplace=True)
df = df.astype(float)  # Convert to float

In [ ]:
df['num'] = df['num'].apply(lambda x: 0 if x == 0 else 1)

In [ ]:
categorical_cols = ['sex', 'cp', 'fbs', 'restecg', 'exang', 'slope', 'ca', 'thal']
le = LabelEncoder()
for col in categorical_cols:
    df[col] = le.fit_transform(df[col])

In [ ]:
X = df.drop('num', axis=1)
y = df['num']
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

In [ ]:
model = keras.Sequential([
    layers.Dense(64, activation='relu', input_shape=(X_train.shape[1],)),  # Input layer
    layers.Dropout(0.2),  # Prevent overfitting
    layers.Dense(32, activation='relu'),  # Hidden layer
    layers.Dense(1, activation='sigmoid')  # Output: 0 or 1
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
history = model.fit(X_train, y_train, epochs=50, batch_size=16, validation_split=0.2)

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [ ]:
model.save('heart_model.h5')

In [ ]:
from flask import Flask, request, render_template_string
from pyngrok import ngrok
import threading  # For running Flask in a thread

# Recompile the model to suppress the warning (optional but recommended)
try:
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    print("Model compiled successfully.")
except Exception as e:
    print("Model compilation error:", str(e))

app = Flask(__name__)

# HTML for input page (now professional and colorful) - unchanged
input_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>CVD Risk Predictor</title>
    <style>
        body { font-family: Arial, sans-serif; background: linear-gradient(to bottom, #e3f2fd, #ffffff); margin: 0; padding: 20px; color: #333; }
        .container { max-width: 600px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); }
        h1 { text-align: center; color: #1976d2; margin-bottom: 20px; }
        form { display: flex; flex-direction: column; }
        label { margin-bottom: 5px; font-weight: bold; color: #424242; }
        input { padding: 10px; margin-bottom: 15px; border: 1px solid #ccc; border-radius: 5px; font-size: 16px; }
        input[type="submit"] { background: #4caf50; color: white; border: none; cursor: pointer; font-size: 18px; padding: 12px; border-radius: 5px; transition: background 0.3s; }
        input[type="submit"]:hover { background: #388e3c; }
        footer { text-align: center; margin-top: 20px; font-size: 14px; color: #666; }
    </style>
</head>
<body>
    <div class="container">
        <h1>🫀 Cardiovascular Disease Risk Predictor</h1>
        <p style="text-align: center; color: #666;">Enter patient details below to assess CVD risk. This tool is for educational purposes only—consult a healthcare professional for advice.</p>
        <form action="/predict" method="post">
            <label for="age">Age:</label>
            <input type="number" id="age" name="age" placeholder="e.g., 50" required>

            <label for="sex">Sex (0=Female, 1=Male):</label>
            <input type="number" id="sex" name="sex" placeholder="0 or 1" required>

            <label for="cp">Chest Pain Type (0-3):</label>
            <input type="number" id="cp" name="cp" placeholder="0-3" required>

            <label for="trestbps">Resting Blood Pressure:</label>
            <input type="number" id="trestbps" name="trestbps" placeholder="e.g., 120" required>

            <label for="chol">Cholesterol:</label>
            <input type="number" id="chol" name="chol" placeholder="e.g., 200" required>

            <label for="fbs">Fasting Blood Sugar (0=No, 1=Yes):</label>
            <input type="number" id="fbs" name="fbs" placeholder="0 or 1" required>

            <label for="restecg">Resting ECG (0-2):</label>
            <input type="number" id="restecg" name="restecg" placeholder="0-2" required>

            <label for="thalach">Max Heart Rate:</label>
            <input type="number" id="thalach" name="thalach" placeholder="e.g., 150" required>

            <label for="exang">Exercise Angina (0=No, 1=Yes):</label>
            <input type="number" id="exang" name="exang" placeholder="0 or 1" required>

            <label for="oldpeak">Oldpeak:</label>
            <input type="number" id="oldpeak" name="oldpeak" placeholder="e.g., 1.0" step="0.1" required>

            <label for="slope">Slope (0-2):</label>
            <input type="number" id="slope" name="slope" placeholder="0-2" required>

            <label for="ca">CA (0-3):</label>
            <input type="number" id="ca" name="ca" placeholder="0-3" required>

            <label for="thal">Thal (0-2):</label>
            <input type="number" id="thal" name="thal" placeholder="0-2" required>

            <input type="submit" value="Predict Risk">
        </form>
        <footer>© 2023 CVD Predictor | Educational Tool</footer>
    </div>
</body>
</html>
"""

# HTML for output page (now professional, colorful, with suggestions and stats) - unchanged
output_html = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Prediction Result</title>
    <style>
        body { font-family: Arial, sans-serif; background: linear-gradient(to bottom, #e8f5e8, #ffffff); margin: 0; padding: 20px; color: #333; }
        .container { max-width: 700px; margin: 0 auto; background: white; padding: 30px; border-radius: 10px; box-shadow: 0 4px 8px rgba(0,0,0,0.1); }
        h1 { text-align: center; color: #1976d2; margin-bottom: 20px; }
        .result { text-align: center; font-size: 24px; font-weight: bold; margin: 20px 0; padding: 15px; border-radius: 5px; }
        .high-risk { background: #ffebee; color: #d32f2f; border: 1px solid #d32f2f; }
        .low-risk { background: #e8f5e8; color: #388e3c; border: 1px solid #388e3c; }
        .stats { background: #f3e5f5; padding: 15px; border-radius: 5px; margin: 20px 0; }
        .suggestions { background: #e3f2fd; padding: 15px; border-radius: 5px; margin: 20px 0; }
        ul { list-style-type: disc; margin-left: 20px; }
        a { display: inline-block; margin-top: 20px; padding: 10px 20px; background: #1976d2; color: white; text-decoration: none; border-radius: 5px; transition: background 0.3s; }
        a:hover { background: #1565c0; }
        footer { text-align: center; margin-top: 20px; font-size: 14px; color: #666; }
    </style>
</head>
<body>
    <div class="container">
        <h1>🫀 CVD Risk Prediction Result</h1>
        <div class="result {{ 'high-risk' if 'High' in result else 'low-risk' }}">
            {{ '⚠️ ' if 'High' in result else '✅ ' }}{{ result }}
        </div>

        <div class="stats">
            <h3>📊 General Statistics</h3>
            <p>Cardiovascular disease (CVD) is a leading cause of death worldwide. According to the World Health Organization (WHO), CVD affects approximately 17.9 million people annually, accounting for 31% of all global deaths. Risk factors include high blood pressure, cholesterol, and lifestyle habits.</p>
        </div>

        <div class="suggestions">
            <h3>💡 Suggestions and Precautions</h3>
            {% if 'High' in result %}
                <p><strong>Since your risk is high, take immediate action:</strong></p>
                <ul>
                    <li>Consult a cardiologist or healthcare provider for a thorough check-up and personalized advice.</li>
                    <li>Monitor and manage blood pressure, cholesterol, and blood sugar levels regularly.</li>
                    <li>Adopt a heart-healthy diet: Reduce salt, saturated fats, and processed foods; increase fruits, vegetables, and whole grains.</li>
                    <li>Engage in regular physical activity: Aim for at least 150 minutes of moderate exercise per week (e.g., brisk walking).</li>
                    <li>Avoid smoking and limit alcohol intake.</li>
                    <li>Track symptoms like chest pain or shortness of breath and seek emergency care if needed.</li>
                </ul>
            {% else %}
                <p><strong>Your risk is low—maintain it with these habits:</strong></p>
                <ul>
                    <li>Schedule annual check-ups to monitor heart health.</li>
                    <li>Eat a balanced diet rich in nutrients and maintain a healthy weight.</li>
                    <li>Stay active: Incorporate daily exercise to keep your heart strong.</li>
                    <li>Avoid risk factors like smoking and excessive stress.</li>
                    <li>Stay informed: Learn about heart health through reliable sources like the American Heart Association.</li>
                </ul>
            {% endif %}
            <p><em>Note: This is general guidance. Always consult a doctor for medical advice.</em></p>
        </div>

        <a href="/">Predict Again</a>
        <footer>© 2023 CVD Predictor | Educational Tool</footer>
    </div>
</body>
</html>
"""

@app.route('/')
def home():
    print("Home route accessed.")
    return render_template_string(input_html)

@app.route('/predict', methods=['POST'])
def predict():
    print("Predict route called.")
    try:
        print("Starting data collection...")
        data = []
        required_keys = ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']
        for key in required_keys:
            value = request.form.get(key, '').strip()
            print(f"Key: {key}, Value: '{value}'")
            if not value:
                raise ValueError(f"Missing value for {key}")
            data.append(float(value))
        print("Data collected:", data)

        print("Checking scaler and model...")
        if 'scaler' not in globals():
            raise NameError("Scaler not loaded. Please run preprocessing cells.")
        if 'model' not in globals():
            raise NameError("Model not loaded. Please run training cells.")

        print("Scaling data...")
        data_scaled = scaler.transform([data])
        print("Scaled data shape:", data_scaled.shape)

        print("Checking for hybrid model...")
        if hasattr(model, 'layers') and any('conv' in str(layer).lower() for layer in model.layers):
            data_scaled = data_scaled.reshape(1, len(data), 1)
            print("Reshaped for hybrid model.")

        print("Making prediction...")
        prediction = model.predict(data_scaled)[0][0]
        print("Prediction value:", prediction)
        result = "High Risk of CVD" if prediction > 0.5 else "Low Risk of CVD"
        print("Result:", result)

        return render_template_string(output_html, result=result)

    except Exception as e:
        print("Error in predict route:", str(e))
        error_msg = f"An error occurred: {str(e)}. Check Colab logs for details."
        error_html = f"""
        <!DOCTYPE html>
        <html>
        <head><title>Error</title></head>
        <body style="font-family: Arial; text-align: center; padding: 50px;">
            <h1>⚠️ Error</h1>
            <p>{error_msg}</p>
            <a href="/">Go Back</a>
        </body>
        </html>
        """
        return error_html

# Function to run Flask in a thread
def run_app():
    print("Starting Flask app on port 5000...")
    app.run(port=5000, debug=False, use_reloader=False)

# Kill existing tunnels
ngrok.kill()
print("Ngrok tunnels killed.")

# Start ngrok tunnel
ngrok.set_auth_token('349K4TFgJpt7It51Ny83YVHP96U_39ZDgKxtjqYjr11RS9iHi')  # Replace with your actual ngrok auth token (get it from ngrok.com)
try:
    public_url = ngrok.connect(5000)
    print("Public URL:", public_url)  # This will print the link to access the app
except Exception as e:
    print("Ngrok error:", str(e))

# Run Flask in a background thread
thread = threading.Thread(target=run_app)
thread.start()

print("Flask app is running in the background. You can now use other cells or access the public URL.")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score
y_pred_prob = model.predict(X_test)  # Get probabilities
y_pred = (y_pred_prob > 0.5).astype(int)
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_pred_prob))

In [ ]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state=42)
X_train_aug, y_train_aug = smote.fit_resample(X_train, y_train)
# Retrain the model with augmented data
model.fit(X_train_aug, y_train_aug, epochs=50, batch_size=16, validation_split=0.2)

In [ ]:
# Reshape for CNN (add a dimension for "time steps")
X_train_reshaped = X_train_aug.reshape(X_train_aug.shape[0], X_train_aug.shape[1], 1)
X_test_reshaped = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

model = keras.Sequential([
    layers.Conv1D(32, kernel_size=3, activation='relu', input_shape=(X_train_reshaped.shape[1], 1)),
    layers.MaxPooling1D(pool_size=2),
    layers.LSTM(32, return_sequences=False),
    layers.Dense(16, activation='relu'),
    layers.Dropout(0.2),
    layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train_reshaped, y_train_aug, epochs=50, batch_size=16, validation_split=0.2)
# Save and update the web app to use this new model
model.save('cvd_model_hybrid.h5')

In [ ]:
import shap
import numpy as np

# Use KernelExplainer which is more suitable for deep learning models
# Wrap model.predict in a lambda function to reshape input to 3D
explainer = shap.KernelExplainer(lambda x: model.predict(np.expand_dims(x, axis=-1)), X_train_reshaped.squeeze())

# Calculate SHAP values for the first 5 test samples
# Remove the extra dimension from X_test_reshaped[:5] before passing to explainer
shap_values = explainer.shap_values(X_test_reshaped[:5].squeeze())

# Reshape shap_values for plotting if necessary (depending on model output shape)
# Assuming a single output for binary classification, shap_values will be a list of arrays.
# We need to take the values corresponding to the positive class (index 1)
if isinstance(shap_values, list):
    shap_values = shap_values[0] # Or shap_values[1] if the model outputs probabilities for both classes

# shap_values should already have the correct shape after squeezing the input to explainer.shap_values
# No need to squeeze shap_values again here.

shap.summary_plot(shap_values, X_test_reshaped[:5].squeeze(), feature_names=column_names[:-1])

In [ ]:
# Test with sample data
sample_data = [50, 1, 3, 145, 233, 1, 0, 150, 0, 2.3, 0, 0, 1]  # Example inputs
try:
    data_scaled = scaler.transform([sample_data])
    if hasattr(model, 'layers') and any('conv' in str(layer).lower() for layer in model.layers):
        data_scaled = data_scaled.reshape(1, len(sample_data), 1)
    prediction = model.predict(data_scaled)[0][0]
    result = "High Risk of CVD" if prediction > 0.5 else "Low Risk of CVD"
    print("Prediction:", result)
except Exception as e:
    print("Error:", str(e))

In [ ]:
!git config --global user.name "manishashivarathri2004-ctrl"
!git config --global user.email "manishashivarathri2004@gmail.com"
